# 01 — Stylized facts of equity returns

**Companion:** [docs/concept-guide.md](../docs/concept-guide.md) §§1–3 · **Timeboxes:** S1–S3 in [ROADMAP.md](../ROADMAP.md)

**Goal:** verify four empirical regularities on ~20 years of S&P 500 daily returns, then build the rolling-volatility estimator that GARCH will replace tomorrow.

The rules:

1. **You write every analysis cell.** Cells marked `TODO(you)` are yours; each states its DONE criterion.
2. **Checkpoints are your oracle.** After each task, compare your numbers against the expected magnitudes. Outside the range → debug before moving on.
3. **Hints are folded and tiered** (concept → API → pseudocode). Open them deliberately, one rung at a time — the struggle is the learning.
4. **After each checkpoint, write the one interpretation sentence it asks for.** On Sunday your README assembles itself from these.

In [1]:
# GIVEN — the only free code in this notebook.
# autoreload: your edits to src/garch_lab/*.py take effect here without restarting the kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scipy
import plotly.graph_objects as go          # go.Figure / go.Scatter / go.Bar — your plotting toolkit
from plotly.subplots import make_subplots  # for the two-panel figures


import os
path_here = os.getcwd()
project_path = os.path.dirname(path_here)
os.chdir(project_path)


# Interactive in the notebook:  fig.show()
# Static PNG for the README:    fig.write_image("figures/figN.png")   (kaleido is installed)

from garch_lab.data import load_prices

px = load_prices("^GSPC", start="2005-01-01")  # price series (kept as `px`); downloads once, then reads data/*.csv
px.tail()

date
2026-07-08    7482.709961
2026-07-09    7543.640137
2026-07-10    7575.390137
2026-07-13    7515.339844
2026-07-14    7547.459961
Name: ^GSPC, dtype: float64

## §1 From prices to returns — guide §1

Prices are non-stationary: statistics computed on them do not converge to anything interpretable. Returns are approximately stationary and scale-free — *they* are the object of study.

**Think before coding:**

- Q1. What would the sample mean of the S&P's price *level* over 2005–2026 actually estimate? Why is stationarity the license to do statistics at all? (You know this from statistical mechanics: which property lets a single realized history stand in for the ensemble?)
- Q2. Log vs simple returns: what exactly do you gain on the time axis, and what do you give up across assets? (Guide §1.2.)

**Task** — implement `log_returns` in [src/garch_lab/returns.py](../src/garch_lab/returns.py).

**DONE when:** `pytest tests/test_returns.py` is green. Then compute `r` and make two stacked plots: price level on top, daily log returns below.

<details><summary>Hint 1 — concept</summary>

A log return is a first difference of log prices. Decide which date should label each return — the test has an opinion, and so does every merge you will ever do against these dates.

</details>

<details><summary>Hint 2 — API</summary>

`np.log`, `Series.diff()`, `Series.dropna()`.

</details>

In [2]:
# TODO(you): make tests/test_returns.py green first, then:
#   r = log_returns(px)
#   two stacked plots: price level (top), daily log returns (bottom)
# DONE when: the tests pass and you can SEE 2008 and 2020 in the returns panel

from garch_lab.returns import log_returns

r = log_returns(px)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Price Level", "Daily Log Returns")
)

# Top: price level
fig.add_trace(
    go.Scatter(
        x=px.index,
        y=px,
        mode="lines",
        name="Price"
    ),
    row=1,
    col=1
)

# Bottom: log returns
fig.add_trace(
    go.Scatter(
        x=r.index,
        y=r,
        mode="lines",
        name="Log Return"
    ),
    row=2,
    col=1
)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Log Return", row=2, col=1)

fig.update_layout(
    height=700,
    showlegend=False,
    title="Price and Daily Log Returns"
)

fig.show()


### ✓ Checkpoint 1

- Observations: n ≈ 5,400 ± 100 (grows with your download date).
- Daily σ̂ ≈ 0.8–1.4% · daily mean ≈ 0.02–0.05% — note the order-of-magnitude gap; §3 will use it.
- The return plot already shows 2008 and 2020 as *clusters* of large bars, of both signs.

➜ **One sentence:** what do you see in the return plot that an iid model forbids?

_Your sentence:_ I see that the returns seem to be dependent on the last day's returns as the "heavy volatility" days are concentrated on the same years (2008 and 2020).

## §2.1 Fat tails — guide §2.1

**Definition — excess kurtosis.** $\kappa = \dfrac{E[(X-\mu)^4]}{\sigma^4} - 3$, the fourth standardized moment minus 3 (the Gaussian's value), so $\kappa = 0$ for a normal distribution.

**Definition — Jarque–Bera statistic.** $JB = \dfrac{n}{6}\left(S^2 + \dfrac{\kappa^2}{4}\right)$, combining sample skewness $S$ and excess kurtosis $\kappa$; under normality $JB \sim \chi^2_2$.

**Predict before you compute:**

- Q1. With your daily σ̂, how many |r| > 4σ days should a *Gaussian* produce in ~5,300 observations? (P(|Z| > 4) ≈ 6.3×10⁻⁵.) Write your number down, then count the real ones.
- Q2. Kurtosis is a fourth-moment estimator. What dominates its own sampling variance — and what does that predict about its stability across subsamples?

**Task:** (a) excess kurtosis on the full sample and on each half separately; (b) count days with |r| > 4σ̂; (c) QQ plot of *standardized* returns vs the normal → export `figures/fig2_qq_tails.png`; (d) fit a Student-t by maximum likelihood, report ν̂; (e) Jarque–Bera.

**DONE when:** the checkpoint magnitudes hit and the figure is exported.

<details><summary>Hint 1 — concept</summary>

Standardize returns (mean 0, variance 1) before the QQ plot so the 45° line is the Gaussian reference.

</details>

<details><summary>Hint 2 — API traps</summary>

</details>

`scipy.stats.kurtosis` already returns EXCESS kurtosis — do not subtract 3 twice. `scipy.stats.t.fit` returns `(df, loc, scale)`, and `scale` is not σ (Var = scale²·ν/(ν−2)). There is no plotly QQ helper: `scipy.stats.probplot` hands back the ordered (theoretical, empirical) quantile pairs — scatter them with `go.Scatter` and add the 45° reference line yourself. `scipy.stats.jarque_bera`.

In [3]:
# TODO(you): kurtosis (full + two halves), 4-sigma day count,
#            QQ plot -> figures/fig2_qq_tails.png, t-fit nu-hat, Jarque-Bera

# Q1: Clearly 6.3*10^-5 * 5300 < 1, thus 0 days.
# Taking σ̂ = 1,2%, we have 4σ̂ = 5% and there are quite a few days with higher (absolute) returns than that.

# Q2: The eigth moment is the samiling variance of the kurtosis, so any outlier (2008, 2020) will dominate.
# This makes it highly dependent on the sample (so pretty much unstable).

In [4]:
# (a) excess kurtosis on the full sample and on each half separately

full_kurtosis = scipy.stats.kurtosis(r)
full_skewness = scipy.stats.skew(r)
first_half_kurtosis = scipy.stats.kurtosis(r[:len(r)//2])
second_half_kurtosis = scipy.stats.kurtosis(r[len(r)//2:])

print("Full sample kurtosis:", full_kurtosis)
print("First half kurtosis:", first_half_kurtosis)
print("Second half kurtosis:", second_half_kurtosis)
print("Full sample skewness:", full_skewness)

# There's quite some difference!

Full sample kurtosis: 13.22008496684579
First half kurtosis: 10.9392746492242
Second half kurtosis: 16.32495485464567
Full sample skewness: -0.47644298746787705


In [5]:
# (b) count days with |r| > 4σ̂;

full_volatility = scipy.stats.tstd(r)

print("Full sample volatility (std dev):", full_volatility)
print("Number of days with |r| > 4σ:", r[abs(r) > 4 * full_volatility].count())
# This is way over 0.

Full sample volatility (std dev): 0.012015264203603162
Number of days with |r| > 4σ: 43


In [6]:
# (c) QQ plot of standardized returns vs the normal

r_ordered = scipy.stats.probplot(r, sparams=(scipy.stats.tmean(r), scipy.stats.tstd(r)))

# Create scatter plot

fig = go.Figure()

# Create scatter plot
fig.add_trace(
    go.Scatter(
        x=r_ordered[0][0],
        y=r_ordered[0][1],
        mode='markers',
        name='QQ Plot'
    )
)

# Add the ideal line
min_val = min(r_ordered[0][1].min(), r_ordered[0][0].min())
max_val = max(r_ordered[0][1].max(), r_ordered[0][0].max())
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        name='Ideal Line',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="QQ plot: daily log returns vs Normal (return units)",
                   xaxis_title="Theoretical quantiles", yaxis_title="Empirical quantiles")


fig.show()

# Export to figures/fig2_qq_tails.png
fig.write_image("figures/fig2_qq_tails.png")

In [7]:
# (d) fit a Student-t by maximum likelihood, report ν̂

df, loc, scale = scipy.stats.t.fit(r)

estimated_t_sigma = scale * np.sqrt(df / (df - 2))

print("Estimated degrees of freedom (ν):", df)
print("Estimated σ:", estimated_t_sigma)


# returns `(df, loc, scale)`, and `scale` is not σ (Var = scale²·ν/(ν−2))

Estimated degrees of freedom (ν): 2.4290447188646196
Estimated σ: 0.015305178703730106


In [8]:
# (e) Jarque–Bera

jb = scipy.stats.jarque_bera(r)

print("Jarque-Bera test statistic:", jb.statistic)
print("Jarque-Bera p-value:", jb.pvalue)
print("The JB >> 1 and its p-value << 0.05, thus we reject the hypothesis of normality for the log returns.")

Jarque-Bera test statistic: 39630.17280329012
Jarque-Bera p-value: 0.0
The JB >> 1 and its p-value << 0.05, thus we reject the hypothesis of normality for the log returns.


### ✓ Checkpoint 2

- κ_excess ∈ [5, 30], and visibly different between the two halves — **that instability is a finding about the tails, not a bug in your code.**
- 4σ days: the Gaussian expects < 1 in this sample; you should count ≥ 10.
- ν̂ ∈ [3, 6] — heavy enough that the fourth moment is borderline undefined, consistent with the instability you just saw.
- QQ: the S-curve — below the line lower-left, above it upper-right.
- Jarque–Bera p ≈ 0.

➜ **One sentence:** why is that p-value nearly uninformative at n ≈ 5,000, and what carries the information instead? (Guide §2.1.)

_Your sentence:_ The p-value is nearly uninformative because the sample size is large enough that even small deviations from normality will lead to a rejection of the null hypothesis. Instead, the information is carried by the magnitude of the test statistic and the observed kurtosis and skewness values, which indicate the presence of heavy tails and asymmetry in the return distribution.

## §2.2 No linear autocorrelation in returns — guide §2.2

**Definition — Autocorrelation function (ACF).** For a (weakly) stationary series $\{X_t\}$ with mean $\mu = E[X_t]$ and variance $\gamma_0 = \mathrm{Var}(X_t)$, the autocorrelation at lag $k$ is

$$
\rho(k) = \frac{\gamma_k}{\gamma_0}, \qquad \gamma_k = \mathrm{Cov}(X_t, X_{t+k}) = E\big[(X_t - \mu)(X_{t+k} - \mu)\big].
$$

The sample estimator used in practice, for a series of length $n$, is

$$
\hat{\rho}(k) = \frac{\sum_{t=1}^{n-k} (X_t - \bar{X})(X_{t+k} - \bar{X})}{\sum_{t=1}^{n} (X_t - \bar{X})^2}.
$$

Under the null hypothesis that $\{X_t\}$ is iid (white noise), $\hat{\rho}(k)$ is asymptotically $\mathcal{N}(0, 1/n)$ for each fixed $k > 0$, which is where the $\pm 1.96/\sqrt{n}$ band (a 95% pointwise confidence interval under that null) comes from.

- Q. Suppose ρ(1) were reliably +0.10. What trade exists, who runs it, and what does their activity do to ρ(1)?

**Task:** ACF of r out to lag 50, with the ±1.96/√n band.

**DONE when:** the plot exists and you can defend what the band assumes.

<details><summary>Hint — API</summary>

`statsmodels.tsa.stattools.acf(r, nlags=50)` returns the ACF values themselves; drop lag 0 (it is 1.0 by definition and flattens the rest), draw the remainder with `go.Bar`, and add the ±1.96/√n band as two horizontal lines (`fig.add_hline`). Computing that band yourself — rather than letting a helper draw it — is where the learning is.

</details>

In [9]:
# TODO(you): ACF of r to lag 50, iid confidence band
import statsmodels.api as sm
acf = sm.tsa.stattools.acf(r, nlags=50)[1:] # Exclude lag 0

# Plotting the ACF with confidence bands

fig = go.Figure()

# Create bar plot
fig.add_trace(
    go.Bar(
        x=np.arange(1, len(acf)+1),
        y=acf,
        name='ACF'
    )
)

# Add 95% confidence bands for iid
conf_band = 1.96/np.sqrt(len(r))
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(acf)+1),
        y=[conf_band]*len(acf),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(acf)+1),
        y=[-conf_band]*len(acf),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="ACF of daily log returns",
                   xaxis_title="Lag", yaxis_title="ACF")


fig.show()
# TODO: I do not fully understand why the first lag is particularily significant.

### ✓ Checkpoint 3

- Essentially all lags inside the band; scattered marginal violations at roughly the 5% rate chance predicts.
- Worth writing down: the band assumes iid — heteroskedasticity (next section!) inflates the true sampling variance of ρ̂, so do not interpret stragglers.

➜ **One sentence:** "uncorrelated is not independent" — where could structure still hide?

_Your sentence:_ There may be second (or higher) order effects. Correlation is just fist order!

## §2.3 Volatility clustering — guide §2.3

- Q. The series r and r² carry exactly the same digits. How can one be white noise while the other has 100-day memory? In which *moment* does the dependence live — and what does that say about the decomposition r_t = σ_t · z_t?

**Task:** ACF of r² and of |r| out to lag 100. Then a two-panel figure — daily returns on top, ACF of |r| below → export `figures/fig1_clustering.png`.

**DONE when:** exported and the checkpoint hits.

In [10]:
# TODO(you): ACF of r^2 and |r| to lag 100;
#            two-panel figure -> figures/fig1_clustering.png

# Q. Actually they do not carry the same information, the sign is not present in r2.
# This hints at the fact that magnitude can be predicted while sign can't (random walk)

acf_r2 = sm.tsa.stattools.acf(r**2, nlags=100)[1:] # Exclude lag 0
acf_abs_r = sm.tsa.stattools.acf(np.abs(r), nlags=100)[1:] # Exclude lag 0

# Plotting the ACF with confidence bands

# Plotting the ACF with confidence bands

fig = go.Figure()

# Create bar plot
fig.add_trace(
    go.Bar(
        x=np.arange(1, len(acf_r2)+1),
        y=acf_r2,
        name='ACF r2'
    )
)

# Create bar plot
fig.add_trace(
    go.Bar(
        x=np.arange(1, len(acf_abs_r)+1),
        y=acf_abs_r,
        name='ACF |r|'
    )
)

# Add 95% confidence bands for iid
conf_band = 1.96/np.sqrt(len(r))
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(acf_r2)+1),
        y=[conf_band]*len(acf_r2),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="ACF of daily log returns squared and absolute value",
                   xaxis_title="Lag", yaxis_title="ACF")

fig.show()

# Export to figures/fig1_clustering.png
fig.write_image("figures/fig1_clustering.png")


### ✓ Checkpoint 4

- ρ̂(1) of |r| ≈ 0.2–0.35; positive and significant out to 50–100+ lags; decaying slowly.
- **This plot is the empirical license for the entire project** — the second moment is forecastable. Your README needs this figure.

➜ **One sentence:** Mandelbrot said it in 1963 — say it in your own words.

_Your sentence:_ Today's returns are similar to tomorrow's returns in size. We just can't predict the sign!

## §2.4 The leverage effect — guide §2.4 · *(cuttable — ROADMAP cut #2)*

- Q. Tomorrow you will feed shocks into GARCH as ε². What information does squaring destroy — and what correlation would demonstrate that the destroyed information actually matters?

**Task:** Corr(r_t, |r_{t+k}|) for k = 1…20, as a bar plot.

<details><summary>Hint — API</summary>

`r.corr(r.abs().shift(-k))` inside a loop or comprehension — mind the direction of the shift: today's return against *future* magnitude.

</details>

In [11]:
# TODO(you): leverage correlations, k = 1..20, bar plot
# Q. The sign, as mentioned earlier.

correlation_r_abs_r = [r.corr(r.abs().shift(-k)) for k in range(1,21)]

# Plotting the correlation

fig = go.Figure()

# Create bar plot
fig.add_trace(
    go.Bar(
        x=np.arange(1, len(correlation_r_abs_r)+1),
        y=correlation_r_abs_r,
        name='Correlation'
    )
)

# Add 95% confidence bands for iid
conf_band = 1.96/np.sqrt(len(r))
fig.add_trace(
    go.Scatter(
        x=np.arange(1, len(correlation_r_abs_r)+1),
        y=[-conf_band]*len(correlation_r_abs_r),
        mode='lines',
        name='Confidence Band',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(title="Correlation of daily log returns with their advanced absolute values",
                   xaxis_title="Days ahead", yaxis_title="Correlation")


fig.show()

### ✓ Checkpoint 5

- Negative at small k, magnitude ≈ −0.05 to −0.15, fading as k grows.

➜ **One sentence:** you have just measured, in advance, the failure mode of the model you will fit tomorrow. Name it.

_Your sentence:_ The sign of the returns affects tomorrow's volatility differently, and a symmetric model fails to take that into account.

## §2.5 Aggregational Gaussianity — guide §2.5 · *(cuttable — ROADMAP cut #1)*

- Q. The CLT grinds aggregated returns toward the Gaussian. Which two features you documented today slow it down?

**Task:** excess kurtosis at 1-day, 5-day, 21-day horizons. Summing log returns over blocks is licensed by the additivity you tested this morning.

<details><summary>Hint — non-overlapping blocks</summary>

Group positions into blocks (`np.arange(len(r)) // 5`) and sum within blocks — check you are not accidentally using overlapping windows, which would smooth the answer.

</details>

In [12]:
# TODO(you): excess kurtosis at 1d / 5d / 21d
for days in [1, 5, 21]:
    r_agg = r.rolling(window=days).sum().dropna()
    kurt = scipy.stats.kurtosis(r_agg)
    print(f"Excess kurtosis for {days}-day aggregated returns: {kurt}")

Excess kurtosis for 1-day aggregated returns: 13.22008496684579
Excess kurtosis for 5-day aggregated returns: 8.716757756095838
Excess kurtosis for 21-day aggregated returns: 9.20790995649966


### ✓ Checkpoint 6

- Kurtosis falls with horizon — toward ≈ 1–5 at 21 days, still above the Gaussian's 0.

➜ **One sentence** tying this to the CLT and to exactly which of its assumptions your data weaken.

_Your sentence:_ We have heavy tails and volatility clustering, which violates the CLT's assumption of independence, slowing down the convergence to a Gaussian distribution.

## §3 Rolling realized volatility — guide §3

**Definition — rolling volatility estimator.** $\hat{\sigma}_t = \sqrt{\dfrac{1}{w}\sum_{i=0}^{w-1} r_{t-i}^2} \times \sqrt{252}$, the annualized root-mean-square of the last $w$ squared returns (a boxcar filter on $r^2$).

- Q1. Your estimator is a boxcar filter on the r² series. What does the window length trade off?
- Q2. Where does √252 come from — and which stylized fact from §2.2 earned you the right to use it?
- Q3. Why is the mean dropped? (Look back at Checkpoint 1.)

**Task** — implement `rolling_vol` in [src/garch_lab/volatility.py](../src/garch_lab/volatility.py).

**DONE when:** `pytest tests/test_volatility.py` is green. Then plot the 21-day annualized vol over the full sample, overlay the 5-day and 63-day variants, and export `figures/fig3_rolling_vol.png` (a draft — tomorrow's version adds the GARCH overlay and takes its README slot).

Vocabulary for your write-up: call yours **"21-day rolling volatility"**, not "realized volatility" — in the literature RV means intraday sums (guide §3.2). Naming it precisely signals you know the difference.

<details><summary>Hint 1 — concept</summary>

Root of the rolling *mean* of squared returns — not a rolling standard deviation. One of the tests exists purely to catch that difference.

</details>

<details><summary>Hint 2 — API</summary>

`Series.rolling(window).mean()` on a squared series, `np.sqrt`, and the constant already defined in the module.

</details>

In [13]:
# TODO(you): green tests; 21d annualized vol plot; overlay 5d and 63d;
#            export figures/fig3_rolling_vol.png

from garch_lab.volatility import rolling_vol

vol_d = [rolling_vol(r, window=window, annualize=True) for window in [5, 21, 63]]

# We plot the three rolling volatilities on the same figure
fig = go.Figure()

for window, vol in zip([5, 21, 63], vol_d):
    fig.add_trace(
        go.Scatter(
            x=vol.index,
            y=vol,
            mode='lines',
            name=f'{window}-day rolling volatility'
        )
    )

fig.update_layout(title="Rolling Volatility of Daily Log Returns",
                   xaxis_title="Date", yaxis_title="Annualized Volatility")

fig.show()

# export figures/fig3_rolling_vol.png
fig.write_image("figures/fig3_rolling_vol.png")

### ✓ Checkpoint 7

- Long-run median ≈ 12–18% annualized; calm 2017 ≈ 6–10%; 2008 and 2020 peaks ≈ 70–90%.
- Debug tells: everything off by ×15.9 → you forgot the square root of 252; off by ×100 → percent vs raw units.

➜ **One sentence:** which window would you hand a risk desk, and what did you trade away to get it?

_Your sentence:_ A risk desk should have access to the 21-day rolling volatility, as it balances responsiveness to market changes with stability. The trade-off is that it may lag in capturing sudden shifts in volatility with respect to the 5-days window.

## Wrap — what you proved today

Collect your seven checkpoint sentences here; they are the draft of the README section "What the data demand of a model" (ROADMAP box S4):

1. I see that the returns seem to be dependent on the last day's returns as the "heavy volatility" days are concentrated on the same years (2008 and 2020).
2. The p-value is nearly uninformative because the sample size is large enough that even small deviations from normality will lead to a rejection of the null hypothesis. Instead, the information is carried by the magnitude of the test statistic and the observed kurtosis and skewness values, which indicate the presence of heavy tails and asymmetry in the return distribution.
3. There may be second (or higher) order effects. Correlation is just fist order!
4. Today's returns are similar to tomorrow's returns in size. We just can't predict the sign!
5. The sign of the returns affects tomorrow's volatility differently, and a symmetric model fails to take that into account.
6. We have heavy tails and volatility clustering, which violates the CLT's assumption of independence, slowing down the convergence to a Gaussian distribution.
7. A risk desk should have access to the 21-day rolling volatility, as it balances responsiveness to market changes with stability. The trade-off is that it may lag in capturing sudden shifts in volatility with respect to the 5-days window.

**Teaser for tomorrow:** your boxcar weights day 21 fully and day 22 at zero, and the window length was chosen by convention, not by the data. GARCH replaces it with an exponentially decaying filter whose decay rate is *estimated by maximum likelihood* — plus mean reversion. Same signal-processing object, better impulse response (guide §3.2 → §4).